In [ ]:
# imports
import pandas as pd
from datasets import load_dataset
from evaluator import Evaluator

In [2]:
# load the Splits! dataset
splits = load_dataset("ecaplan/splits", "splits")
# all versions of the dataset contain only a 'train' portion
splits = splits['train']

In [3]:
# convert the huggingface dataset to a pandas dataframe
df = pd.DataFrame(splits)

In [ ]:
# since the full dataset has >176k samples, we will use a smaller sample for testing
sample_size = 10
# sample the dataframe
sample_df = df.sample(sample_size, random_state=42)
sample_df

In [ ]:
# now let's make some theories!
# in this example, we will make a really simple theory that says that group A uses more commas than group B when talking about the topic
# in general, your method can use the two groups, the topic, and the calibration set to make a theory
theories = []

for i, row in sample_df.iterrows():
    demo_A = row['demographic_A']
    demo_B = row['demographic_B']
    topic = row['description']
    calibration_set = row['example_posts']

    # make a theory for each row
    theory = f"People who are '{row['demographic_A']}' use more commas than people who are '{row['demographic_B']}' when talking about '{row['description']}'."
    theories.append(theory)



sample_df['theory'] = theories
sample_df

In [ ]:
# if quantize=True, it will load the Llama-3.1-70B model in 8-bit
# for our experiments, we used full precision, so we recommend using quantize=False if you have enough memory
quantize = True
# batch size for ~4k tokens (raise at your own risk)
batch_size = 1

# now let's load the classification model (make sure you have access to meta-llama/Llama-3.1-70B-Instruct through huggingface)
evaluator = Evaluator(sample_df, 
                      quantize=quantize,
                      batch_size=batch_size)

In [ ]:
# and let's evaluate the theories! (this can take a while)
evaluator.evaluate()

In [14]:
# now we can see the results according to many splits

# pure accuracy
accuracy = evaluator.get_accuracy()
print(f"Accuracy: {accuracy}")

Accuracy: 0.4


In [9]:
# split by pairs of demographics
evaluator.split_by_demographic_A_demographic_B()

,demographic_A,demographic_B,count,accuracy
0,black,catholic,1,1.000000
1,black,construction,2,0.000000
2,catholic,construction,1,1.000000
3,catholic,teacher,3,0.333333
4,construction,teacher,2,0.500000
5,jewish,teacher,1,0.000000


In [10]:
# by a individual demographic
evaluator.split_by_unique_demographic()

,unique_demographic,count,accuracy
0,teacher,6,0.333333
1,jewish,1,0.000000
2,black,3,0.333333
3,construction,5,0.400000
4,catholic,5,0.600000


In [11]:
# by topic category
evaluator.split_by_category()

,category,count,accuracy
0,Entertainment & Media,1,0.000000
1,Finance & Investing,2,0.000000
2,Hobbies & Special Interests,1,0.000000
3,Professional & Career-oriented Spaces,2,1.000000
4,Sports & Fitness,3,0.666667
5,Technology & Gaming,1,0.000000


In [13]:
# by demographic pair and topic category
evaluator.split_by_category_demographic_A_demographic_B()

,category,demographic_A,demographic_B,count,accuracy
0,Entertainment & Media,construction,teacher,1,0.0
1,Finance & Investing,black,construction,2,0.0
2,Hobbies & Special Interests,catholic,teacher,1,0.0
3,Professional & Career-oriented Spaces,catholic,teacher,1,1.0
4,Professional & Career-oriented Spaces,construction,teacher,1,1.0
5,Sports & Fitness,black,catholic,1,1.0
6,Sports & Fitness,catholic,construction,1,1.0
7,Sports & Fitness,jewish,teacher,1,0.0
8,Technology & Gaming,catholic,teacher,1,0.0


In [15]:
# by individual demographic and topic category
evaluator.split_by_category_unique_demographic()

,category,unique_demographic,count,accuracy
0,Finance & Investing,teacher,0,NaN
1,Finance & Investing,jewish,0,NaN
2,Finance & Investing,black,2,0.0
3,Finance & Investing,construction,2,0.0
4,Finance & Investing,catholic,0,NaN
5,Professional & Career-oriented Spaces,teacher,2,1.0
6,Professional & Career-oriented Spaces,jewish,0,NaN
7,Professional & Career-oriented Spaces,black,0,NaN
8,Professional & Career-oriented Spaces,construction,1,1.0
9,Professional & Career-oriented Spaces,catholic,1,1.0


In [22]:
# by topic and individual demographic
evaluator.split_by_description_unique_demographic()

,description,unique_demographic,count,accuracy
0,Side Hustles/Additional Income,teacher,0,NaN
1,Side Hustles/Additional Income,jewish,0,NaN
2,Side Hustles/Additional Income,black,1,0.0
3,Side Hustles/Additional Income,construction,1,0.0
4,Side Hustles/Additional Income,catholic,0,NaN
5,Medical & Healthcare Professions,teacher,1,1.0
6,Medical & Healthcare Professions,jewish,0,NaN
7,Medical & Healthcare Professions,black,0,NaN
8,Medical & Healthcare Professions,construction,1,1.0
9,Medical & Healthcare Professions,catholic,0,NaN


In [21]:
# by topic and demographic pair
evaluator.split_by_description_demographic_A_demographic_B()

,description,demographic_A,demographic_B,count,accuracy
0,Birdwatching,catholic,teacher,1,0.0
1,"Consoles (PlayStation, Xbox, Nintendo)",catholic,teacher,1,0.0
2,Freelancing,catholic,teacher,1,1.0
3,Index Funds & ETFs,black,construction,1,0.0
4,Medical & Healthcare Professions,construction,teacher,1,1.0
5,Side Hustles/Additional Income,black,construction,1,0.0
6,Sitcoms,construction,teacher,1,0.0
7,Skateboarding,black,catholic,1,1.0
8,Swimming,catholic,construction,1,1.0
9,Yoga,jewish,teacher,1,0.0


In [ ]:
# to get all splits, you can use the following function
all_splits = evaluator.get_all_split_reports()
all_splits